# Driver Behavior Analysis using Smartphone Sensors & Computer Vision

By Ajay Bhagwan Gaikwad

[Open in Colab](https://colab.research.google.com/github/AjayGaikwad-07/driver-behavior-ml/blob/main/driver_behavior_colab_master.ipynb)

---
This notebook is my project on detecting dangerous driver behavior using smartphone sensors (accelerometer, gyroscope, GPS) and some basic computer vision on video. I tried to break it down into stages so it's easy to follow along. Each stage answers a question, does some work, shows results, and then I write what I think it means.

## What this notebook covers (stage by stage):
1. Problem definition - why this matters
2. Understanding the dataset (DriverSVT)
3. Data cleaning and checking for missing values
4. EDA - looking at class imbalance and sensor patterns
5. Feature engineering - making orientation-invariant features
6. Baseline model and why accuracy is misleading here
7. Training models with proper train/test split (no driver leakage)
8. Evaluation and confusion matrix analysis
9. Feature ablation - which features actually help?
10. Video experiment with MediaPipe and YOLO
11. Trip risk scoring (0-100 safety score)
12. Final takeaways and what I'd do next

## Stage 1: Why am I doing this?

So the basic problem is - a lot of road accidents happen because drivers are distracted, tired, or doing stuff like using phone while driving. The fancy driver monitoring systems in cars cost a lot (cameras, steering sensors, etc.) and not everyone can afford them, especially in places like taxi fleets or two-wheelers in developing countries.

My question is: can we just use a normal smartphone's sensors (accelerometer, gyroscope, GPS) plus some lightweight computer vision to detect bad driving behavior? Without needing expensive hardware?

The task is a 9-class classification problem. The classes are: `Normal`, `Distraction`, `Drowsiness`, `Using phone`, `No attention`, `Smoking`, `Eating`, `Unfastened belt`, `Camera off`.

Also I want to do facial fatigue detection (eye openness, mouth openness) and detect objects like phones using video.

One important thing - I'm NOT using raw accuracy as my main metric. If 80% of the data is just 'Normal' driving, a model that always predicts 'Normal' gets 80% accuracy but it completely fails to catch any actual dangerous behavior. That's useless for safety. So I'm using **Macro-F1** as my main metric (treats all 9 classes equally) and balanced accuracy as a secondary one.

## Setup - checking hardware and installing packages

Just setting up the environment first.

In [ ]:
# Check what hardware we're running on
import sys
import torch

print(f"Python: {sys.version.split()[0]}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running on CPU. That's fine for tabular ML, but GPU helps with the video stuff later.")

In [ ]:
# Install everything we need
!pip install -q --upgrade pip
!pip install -q ultralytics mediapipe opencv-python-headless scikit-learn pandas numpy matplotlib seaborn joblib
print("Packages installed!")

## Stage 2: The Dataset

I'm using the **DriverSVT** dataset. It was collected from 633 different drivers using smartphones mounted in their cars. The full dataset is huge (like 17.5 million rows, ~9GB), so for Colab I'm using a balanced sample that's about 580K rows across 491 drivers (~68MB).

The data has sensor readings (speed, g-force, accelerometer, gyroscope) plus some facial proxies (perclos = eye closure %, mouth_open_ratio) and a label column called `dangerousstate`.

Here's the column dictionary:

| Column | What it means | Unit |
|---|---|---|
| `datetime` | timestamp | unix ms |
| `speed` | vehicle speed | km/h |
| `gforce` | g-force | G |
| `acc_X/Y/Z` | linear acceleration | m/s² |
| `accelerometer_data_X/Y/Z` | total acceleration (with gravity) | m/s² |
| `gyroscope_data_X/Y/Z` | angular velocity | deg/s |
| `perclos` | eye closure percentage | 0-1 |
| `mouth_open_ratio` | mouth openness | 0-1 |
| `userid` | driver ID | integer |
| `dangerousstate` | the label (9 classes) | text |

No synthetic data is used anywhere - everything runs on real DriverSVT telemetry.

In [ ]:
from pathlib import Path
import os
import urllib.request
import pandas as pd
import numpy as np

# Clone my repo if it's not already there (for helper scripts and models)
REPO_URL = "https://github.com/AjayGaikwad-07/driver-behavior-ml.git"
REPO_DIR = Path("driver-behavior-ml")

if not REPO_DIR.exists() and not Path("day3_prepare_dataset.py").exists():
    print(f"Cloning repo from {REPO_URL} ...")
    !git clone $REPO_URL
    os.chdir(REPO_DIR)
    print("Now in:", os.getcwd())

# Make sure folders exist
for folder in ["data/demo", "data/raw", "models", "reports"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Download MediaPipe face model if we don't have it yet
TASK_PATH = Path("models/face_landmarker.task")
if not TASK_PATH.exists():
    print("Downloading face_landmarker.task ...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, TASK_PATH)
    print(f"Done! ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")
else:
    print(f"Already have face_landmarker.task ({TASK_PATH.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Find the dataset - check a bunch of possible locations
CANDIDATE_PATHS = [
    Path("day3_balanced_group_split_fast/sampled_driver_data_balanced_group_split.csv"),
    Path("day3_working/sampled_driver_data.csv"),
    Path("data/raw/DriverSVT/DriverSVT.csv"),
    Path("/content/drive/MyDrive/driver-behavior-ai/sampled_driver_data_balanced_group_split.csv"),
    Path("/content/drive/MyDrive/DriverSVT/DriverSVT.csv"),
    Path("sampled_driver_data_balanced_group_split.csv")
]

DATASET_PATH = None
for p in CANDIDATE_PATHS:
    if p.exists():
        DATASET_PATH = p
        break

# If we can't find it, ask the user to upload it
if DATASET_PATH is None:
    print("Couldn't find the dataset. Either mount Drive or upload the CSV.")
    try:
        from google.colab import files
        print("Pick sampled_driver_data_balanced_group_split.csv:")
        uploaded = files.upload()
        if uploaded:
            DATASET_PATH = Path(list(uploaded.keys())[0])
    except Exception:
        raise FileNotFoundError("Please upload the CSV to continue.")

if DATASET_PATH and DATASET_PATH.exists():
    print(f"Loading dataset from: {DATASET_PATH}")
    df = pd.read_csv(DATASET_PATH, low_memory=False)
    print(f"Loaded! {len(df):,} rows, {df.shape[1]} columns, {df['userid'].nunique()} drivers.")
else:
    raise FileNotFoundError("Couldn't find the dataset.")

## Stage 3: Data Quality Check

Before doing anything with the data, I want to check if there are missing values or anything weird. Sensor data can have dropouts because the phone hardware doesn't always poll perfectly.

In [ ]:
RAW_FEATURES = [
    "speed", "gforce", "acc_X", "acc_Y", "acc_Z",
    "accelerometer_data_X", "accelerometer_data_Y", "accelerometer_data_Z",
    "gyroscope_data_X", "gyroscope_data_Y", "gyroscope_data_Z",
    "perclos", "mouth_open_ratio"
]
TARGET = "dangerousstate"
GROUP = "userid"

# Make sure numeric columns are actually numeric
for col in RAW_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Check how many missing values each column has
missing_df = pd.DataFrame({
    "Feature": RAW_FEATURES,
    "Missing_Count": [df[c].isnull().sum() for c in RAW_FEATURES],
    "Missing_Pct": [df[c].isnull().mean() * 100 for c in RAW_FEATURES]
})

print("Missing values in each feature:")
display(missing_df.round(4))

So the missing rates are pretty low (around 0.2-0.3% for the sensor columns). This is normal for phone sensor data - sometimes the hardware just drops a reading. I'm not going to throw away those rows because in real-world driving, sensor dropouts happen all the time and the model needs to handle them. Instead I'll use median imputation in the pipeline later so missing values get filled automatically.

## Stage 4: EDA - Looking at the data

Two things I want to check here:
1. How bad is the class imbalance?
2. Do the sensor readings look different for different driving states?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Class distribution
class_counts = df[TARGET].value_counts()
class_pcts = df[TARGET].value_counts(normalize=True) * 100
colors = sns.color_palette("viridis", len(class_counts))

axes[0].barh(class_counts.index, class_counts.values, color=colors)
axes[0].set_title(f"Class Distribution (Total: {len(df):,})", fontsize=12)
axes[0].set_xlabel("Count")
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pcts.values)):
    axes[0].text(cnt + max(class_counts.values)*0.01, i, f"{cnt:,} ({pct:.1f}%)", va="center", fontsize=9)

# 2. Speed distribution for a few key classes
sample_subset = df[df[TARGET].isin(["Normal", "Drowsiness", "Distraction", "Using phone"])].sample(min(10000, len(df)), random_state=42)
sns.boxplot(data=sample_subset, x=TARGET, y="speed", palette="Set2", ax=axes[1])
axes[1].set_title("Speed across different states", fontsize=12)
axes[1].set_xlabel("State")
axes[1].set_ylabel("Speed (km/h)")

plt.tight_layout()
plt.show()

print("Exact numbers:")
display(pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pcts.round(2)}))

What I see here:
- The classes are really imbalanced. Normal and Distraction make up most of the data, while rare stuff like Smoking and Eating are tiny slivers.
- Speed distributions overlap a LOT between Normal and Distraction. You can't just look at speed to tell if someone is distracted - people use their phone at 0 km/h (stopped at a light) and at 80 km/h (highway).
- So the models need class weighting, otherwise they'll just ignore the rare dangerous classes entirely.

## Stage 5: Feature Engineering

Here's a problem I realized - the accelerometer and gyroscope readings depend on how the phone is oriented in the car. If the phone is in a cupholder vs mounted on the dash, the X/Y/Z values are totally different even for the same driving motion.

So I want to make features that don't care about phone orientation:
1. **Total acceleration magnitude** = sqrt(ax² + ay² + az²) - this is the same no matter how the phone is tilted
2. **Total angular velocity magnitude** = sqrt(gx² + gy² + gz²) - same idea for gyroscope
3. **Kinetic intensity** = speed × gforce - this highlights aggressive high-speed maneuvers

In [ ]:
# Make orientation-invariant features
df["acc_total_mag"] = np.sqrt(df["acc_X"]**2 + df["acc_Y"]**2 + df["acc_Z"]**2)
df["gyro_total_mag"] = np.sqrt(df["gyroscope_data_X"]**2 + df["gyroscope_data_Y"]**2 + df["gyroscope_data_Z"]**2)
df["kinetic_intensity"] = df["speed"] * df["gforce"]

ENGINEERED_FEATURES = RAW_FEATURES + ["acc_total_mag", "gyro_total_mag", "kinetic_intensity"]

print(f"Made 3 new features. Total features now: {len(ENGINEERED_FEATURES)}")
df[["speed", "gforce", "acc_total_mag", "gyro_total_mag", "kinetic_intensity"]].head(5)

Let me also check how these engineered features correlate with each other and with the facial proxies.

In [ ]:
corr_cols = ["speed", "gforce", "acc_total_mag", "gyro_total_mag", "kinetic_intensity", "perclos", "mouth_open_ratio"]
corr = df[corr_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, vmin=-1, vmax=1)
plt.title("Correlation Matrix", fontsize=12)
plt.tight_layout()
plt.show()

## Stage 6: Baseline Model (and why accuracy is useless here)

Before training real models, I want to see what a dumb baseline gets. I'm using a DummyClassifier that just predicts the most frequent class every single time. This shows us the floor - the minimum performance you get by doing literally nothing smart.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(df[ENGINEERED_FEATURES], df[TARGET])
dummy_preds = dummy.predict(df[ENGINEERED_FEATURES])

print("=" * 50)
print("  DUMMY BASELINE (always predicts most common class)")
print("=" * 50)
print(f" Accuracy           : {accuracy_score(df[TARGET], dummy_preds):.4f}")
print(f" Balanced Accuracy  : {balanced_accuracy_score(df[TARGET], dummy_preds):.4f}")
print(f" Macro-F1           : {f1_score(df[TARGET], dummy_preds, average='macro', zero_division=0):.4f}")
print("=" * 50)

So the dummy model gets like 27-41% accuracy which sounds decent at first... but look at the Macro-F1 - it's basically zero (around 0.047). And balanced accuracy is only 11% which is just random chance for 9 classes.

This is the 'accuracy paradox'. The model looks okay if you only look at accuracy, but it's actually failing to detect 100% of drowsy drivers, phone users, and smokers. For a safety system that's completely useless. That's why I'm using Macro-F1 as my main metric instead.

## Stage 7: Training Models (the right way)

This is a really important part. If I just do a normal train_test_split, rows from the same driver will end up in both train and test sets. That's bad because the same driver has the same car vibration, same phone mount angle, same driving habits - so the model 'memorizes' those instead of actually learning driving behavior. This is called data leakage and it gives you fake inflated scores.

Instead I'm using GroupShuffleSplit on the userid column. This makes sure every driver in the test set was NEVER seen during training. This gives us the true real-world performance.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# If the balanced dataset already has a split column, use it
if "split" in df.columns:
    train_df = df[df["split"] == "train"].copy()
    test_df = df[df["split"] == "test"].copy()
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df[GROUP]))
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()
    train_df["split"] = "train"
    test_df["split"] = "test"

train_drivers = set(train_df[GROUP].unique())
test_drivers = set(test_df[GROUP].unique())
driver_leakage = train_drivers.intersection(test_drivers)

print("=" * 60)
print(f"Train: {len(train_df):,} rows, {len(train_drivers)} drivers")
print(f"Test : {len(test_df):,} rows, {len(test_drivers)} drivers")
print(f"Driver overlap (should be 0): {len(driver_leakage)}")
print("=" * 60)

assert len(driver_leakage) == 0, "Data leakage! Same drivers in train and test!"
print("Good - no leakage. Drivers are fully separated.")

Now I'll train a few different models and compare them. I'm using median imputation (to fill missing values) and class_weight balancing (so rare classes don't get ignored). The models I'm trying:
1. Logistic Regression - simple linear baseline
2. Random Forest - ensemble of decision trees
3. ExtraTrees - like random forest but more randomized
4. HistGradientBoosting - gradient boosted trees

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_train = train_df[ENGINEERED_FEATURES]
y_train = train_df[TARGET].astype(str)
X_test = test_df[ENGINEERED_FEATURES]
y_test = test_df[TARGET].astype(str)

models_to_evaluate = {
    "Majority Baseline": DummyClassifier(strategy="most_frequent"),
    
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=400, class_weight="balanced", random_state=42))
    ]),
    
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=80, max_depth=16, min_samples_leaf=4,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42
        ))
    ]),
    
    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=80, max_depth=16, min_samples_leaf=4,
            class_weight="balanced", n_jobs=-1, random_state=42
        ))
    ])
}

fitted_models = {}
benchmark_results = []

print("Training models...")
for name, model in models_to_evaluate.items():
    print(f"  Training {name} ...")
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0
    
    preds = model.predict(X_test)
    fitted_models[name] = model
    
    benchmark_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Balanced Accuracy": balanced_accuracy_score(y_test, preds),
        "Macro-F1": f1_score(y_test, preds, average="macro", zero_division=0),
        "Weighted-F1": f1_score(y_test, preds, average="weighted", zero_division=0),
        "Fit Time (s)": round(fit_time, 2)
    })

benchmark_df = pd.DataFrame(benchmark_results).sort_values(by="Macro-F1", ascending=False)
display(benchmark_df.round(4))

## Stage 8: Looking at where the model gets confused

Now I want to dig into the best model (Random Forest) and see which classes it's good at and which ones it struggles with. A confusion matrix is the best way to see this.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_rf = fitted_models["Random Forest"]
rf_preds = best_rf.predict(X_test)
classes = sorted(y_test.unique())

print("Classification report (on unseen drivers):")
print(classification_report(y_test, rf_preds, digits=4, zero_division=0))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, rf_preds, labels=classes, normalize="true")
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix (Random Forest, unseen drivers)", fontsize=12)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

What I noticed from the confusion matrix:
1. Drowsiness detection is actually okay (around 45-55% recall) because the model uses perclos (eye closure) plus the fact that drowsy drivers don't react as much to road conditions.
2. Smoking and Using phone are really hard to detect. This makes sense - if the phone is mounted on the dash, the driver's hand movements inside the cabin barely move the phone's accelerometer at all.
3. The big takeaway: inertial sensors alone can't detect stuff happening inside the cabin. You really need video/camera for that.

## Stage 9: Feature Ablation - which features matter?

I want to know if the facial proxies (perclos, mouth_open_ratio) actually add value on top of the motion sensors. So I'll train the same Random Forest model three times with different feature sets and compare:
1. Motion features only (speed, gforce, acc, gyro, my engineered magnitudes)
2. State proxies only (just perclos and mouth_open_ratio)
3. Combined (everything together)

In [ ]:
MOTION_COLS = [
    "speed", "gforce", "acc_X", "acc_Y", "acc_Z",
    "accelerometer_data_X", "accelerometer_data_Y", "accelerometer_data_Z",
    "gyroscope_data_X", "gyroscope_data_Y", "gyroscope_data_Z",
    "acc_total_mag", "gyro_total_mag", "kinetic_intensity"
]
STATE_COLS = ["perclos", "mouth_open_ratio"]
COMBINED_COLS = MOTION_COLS + STATE_COLS

ablation_configs = {
    "Motion Only": [c for c in MOTION_COLS if c in df.columns],
    "State Only": [c for c in STATE_COLS if c in df.columns],
    "Combined": [c for c in COMBINED_COLS if c in df.columns]
}

ablation_records = []
for group_name, cols in ablation_configs.items():
    print(f"Testing {group_name} ({len(cols)} features)...")
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(n_estimators=60, max_depth=16, class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ])
    pipe.fit(train_df[cols], y_train)
    preds = pipe.predict(test_df[cols])
    
    ablation_records.append({
        "Feature Set": group_name,
        "Feature Count": len(cols),
        "Accuracy": accuracy_score(y_test, preds),
        "Balanced Accuracy": balanced_accuracy_score(y_test, preds),
        "Macro-F1": f1_score(y_test, preds, average="macro", zero_division=0)
    })

ablation_df = pd.DataFrame(ablation_records).sort_values(by="Macro-F1", ascending=False)
display(ablation_df.round(4))

# Bar chart to make it visual
plt.figure(figsize=(8, 4))
sns.barplot(data=ablation_df, x="Feature Set", y="Macro-F1", palette="Blues_d")
plt.title("Which feature set does best on Macro-F1?", fontsize=12)
plt.ylabel("Macro-F1")
plt.ylim(0, max(ablation_df["Macro-F1"]) * 1.3)
for i, v in enumerate(ablation_df["Macro-F1"]):
    plt.text(i, v + 0.005, f"{v:.4f}", ha="center")
plt.tight_layout()
plt.show()

The combined features (motion + state) do the best on Macro-F1. State-only does the worst because eye/mouth openness tells you nothing about vehicle speed or harsh braking. You need both - that's the whole point of a multimodal approach.

## Stage 10: Video Experiment - MediaPipe + YOLO

Now for the fun part - actually running computer vision on a real driving video. I'm using two tools:

1. **MediaPipe Face Landmarker** - detects 468 face landmarks. From those I compute:
   - EAR (Eye Aspect Ratio) - how open the eyes are. Below 0.20 = drowsy
   - MAR (Mouth Aspect Ratio) - how open the mouth is. Above 0.55 = yawning

2. **YOLOv8** - detects objects like cell phone, cup, bottle in the frame

One thing I want to be honest about - if the face isn't visible (driver turned away, bad lighting), I report it as 'Face Not Detected'. I'm NOT going to make up fake data.

In [ ]:
import cv2
import math
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

# Helper functions for computing EAR and MAR from face landmarks
def euclidean_dist(p1, p2):
    return math.hypot(p1.x - p2.x, p1.y - p2.y)

def calculate_ear(pts, eye_idx):
    # EAR = (vertical distances) / (2 * horizontal distance)
    v1 = euclidean_dist(pts[eye_idx[1]], pts[eye_idx[5]])
    v2 = euclidean_dist(pts[eye_idx[2]], pts[eye_idx[4]])
    h  = euclidean_dist(pts[eye_idx[0]], pts[eye_idx[3]])
    return (v1 + v2) / (2.0 * h + 1e-9)

def calculate_mar(pts):
    h = euclidean_dist(pts[13], pts[14])
    w = euclidean_dist(pts[78], pts[308])
    return h / (w + 1e-9)

# MediaPipe landmark indices for left and right eye
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

yolo = YOLO("yolov8n.pt")
print("CV models ready.")

In [ ]:
# Process the demo video frame by frame
VIDEO_PATH = Path("data/demo/driving_demo.mp4")
if not VIDEO_PATH.exists():
    vids = list(Path("data/demo").glob("*.mp4"))
    if vids:
        VIDEO_PATH = vids[0]

video_measurements = []
total_sampled = 0
faces_detected = 0

if VIDEO_PATH.exists() and TASK_PATH.exists():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Processing: {VIDEO_PATH.name} ({total_frames} frames @ {fps:.1f} FPS)")
    
    opts = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(TASK_PATH)),
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1
    )
    
    step = max(1, int(round(fps / 5)))  # sample about 5 frames per second
    frame_no = 0
    
    with vision.FaceLandmarker.create_from_options(opts) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret or frame_no > 300:  # first 300 sampled frames
                break
            frame_no += 1
            if frame_no % step != 0:
                continue
            
            total_sampled += 1
            timestamp_sec = frame_no / fps
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = landmarker.detect_for_video(mp_image, int(timestamp_sec * 1000))
            
            if res.face_landmarks:
                faces_detected += 1
                landmarks = res.face_landmarks[0]
                ear_val = (calculate_ear(landmarks, LEFT_EYE) + calculate_ear(landmarks, RIGHT_EYE)) / 2.0
                mar_val = calculate_mar(landmarks)
                
                video_measurements.append({
                    "timestamp_s": round(timestamp_sec, 2),
                    "face_detected": True,
                    "EAR": ear_val,
                    "MAR": mar_val,
                    "drowsy_flag": ear_val < 0.20,
                    "yawn_flag": mar_val > 0.55
                })
            else:
                # Honestly report no detection
                video_measurements.append({
                    "timestamp_s": round(timestamp_sec, 2),
                    "face_detected": False,
                    "EAR": np.nan,
                    "MAR": np.nan,
                    "drowsy_flag": False,
                    "yawn_flag": False
                })
    cap.release()
    
    video_df = pd.DataFrame(video_measurements)
    detection_rate = (faces_detected / total_sampled) * 100 if total_sampled else 0
    print(f"Done!")
    print(f"  Frames sampled : {total_sampled}")
    print(f"  Faces detected : {faces_detected} ({detection_rate:.1f}%)")
    
    # Plot the results
    plt.figure(figsize=(12, 4))
    valid_pts = video_df.dropna(subset=["EAR", "MAR"])
    if not valid_pts.empty:
        plt.plot(valid_pts["timestamp_s"], valid_pts["EAR"], label="EAR (eye openness)", color="dodgerblue")
        plt.axhline(0.20, color="blue", linestyle="--", alpha=0.7, label="Drowsy threshold (EAR < 0.20)")
        plt.plot(valid_pts["timestamp_s"], valid_pts["MAR"], label="MAR (mouth openness)", color="darkorange")
        plt.axhline(0.55, color="red", linestyle="--", alpha=0.7, label="Yawn threshold (MAR > 0.55)")
        plt.xlabel("Time (seconds)")
        plt.ylabel("Ratio")
        plt.title(f"Driver Alertness from Video ({VIDEO_PATH.name})", fontsize=12)
        plt.legend(loc="upper right")
        plt.tight_layout()
        plt.show()
    else:
        print("No faces were detected in the sampled frames. Reporting this honestly.")
else:
    print(f"No demo video found at {VIDEO_PATH}. Put a driving video in data/demo/ to run this.")
    video_df = pd.DataFrame()

## Stage 11: Trip Risk Score (0-100)

Now I want to turn all these detected events into a single trip safety score from 0 to 100. The idea is simple - you start at 100 and deduct points for each bad thing that was detected:
- Drowsiness episode: -10 points
- Yawning: -4 points
- Harsh maneuver: -8 points
- Phone detected: -15 points

In [ ]:
# Count real events from the video and sensor data
measured_drowsy_events = int(video_df["drowsy_flag"].sum()) if not video_df.empty else 0
measured_yawn_events = int(video_df["yawn_flag"].sum()) if not video_df.empty else 0

# Count harsh kinematic events in the test split
harsh_kinematics = int(((test_df["speed"] > 40) & (test_df["gforce"] > 0.4)).sum())
simulated_trip_harsh = min(harsh_kinematics, 2)  # scale down for a single trip

def compute_safety_score(drowsy_cnt, yawn_cnt, harsh_cnt, phone_cnt=0):
    penalties = {
        "Drowsiness (eyes closed)": drowsy_cnt * 10.0,
        "Yawning": yawn_cnt * 4.0,
        "Harsh maneuvers": harsh_cnt * 8.0,
        "Phone distraction": phone_cnt * 15.0
    }
    total_deductions = sum(penalties.values())
    final_score = max(0.0, min(100.0, 100.0 - total_deductions))
    
    if final_score >= 85:
        rating = "Safe"
    elif final_score >= 70:
        rating = "Moderate caution"
    else:
        rating = "High risk - needs intervention"
    
    return {
        "Trip Safety Score": round(final_score, 1),
        "Rating": rating,
        "Drowsiness episodes": drowsy_cnt,
        "Yawning incidents": yawn_cnt,
        "Harsh maneuvers": harsh_cnt,
        "Total deductions": round(total_deductions, 1),
        "Penalty breakdown": penalties
    }

trip_report = compute_safety_score(
    drowsy_cnt=measured_drowsy_events,
    yawn_cnt=measured_yawn_events,
    harsh_cnt=simulated_trip_harsh,
    phone_cnt=0
)

print("=" * 55)
print("    TRIP RISK ASSESSMENT")
print("=" * 55)
for k, v in trip_report.items():
    if k != "Penalty breakdown":
        print(f"  {k:<25} : {v}")
print("\n  Penalty breakdown:")
for k, v in trip_report["Penalty breakdown"].items():
    print(f"    - {k:<25} : -{v} pts")
print("=" * 55)

## Stage 12: Final Thoughts & Limitations

### What I learned from this project:
1. **Accuracy is a trap on imbalanced data.** The dummy model got ~30-40% accuracy but couldn't detect any actual dangerous behavior. Macro-F1 and balanced accuracy are what matter for safety.
2. **You have to split by driver.** If the same driver shows up in both train and test, you get inflated scores that don't reflect real-world performance. Group splitting on userid is the way to go.
3. **Inertial sensors have limits.** Accelerometer and gyroscope are good for detecting aggressive driving (hard braking, sharp turns) but they can't detect phone usage or smoking when the phone is mounted. Video is needed for that.
4. **Combining modalities works best.** The ablation study showed motion + facial features together beat either one alone.

### Limitations:
- This is a research prototype, not a production safety system. Don't use it for real safety decisions.
- Privacy matters - facial landmarks should be computed on the device, not sent to a server. Raw video shouldn't be stored without the driver's consent.
- The rare classes (Smoking, Eating) are still really hard. Need more data for those or maybe oversampling techniques like SMOTE.

### What I'd do next:
- Export models to ONNX for mobile deployment
- Try temporal models (LSTM or TCN) on sliding windows instead of single-row classification
- Add drift monitoring to detect when the phone sensor calibration changes over time